[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C68_Eval_Infrastructure_Course/01_task_spec/01_task_spec.ipynb)

# 01 · Task spec 与数据集版本化（schema / 稳定 ID / 版本链 / 子集 / 迁移）

目标：把「任务集只增不改」这条纪律，变成**几个可以运行、可以断言的机制**。

本 notebook 你会亲手实现：
1. **spec 的 schema 校验与规范化** —— 填默认值、迁移旧版本，**指纹在规范化之后算**
2. **内容哈希 vs 稳定 ID** —— 改一个错别字，两种方案下历史结果各会怎样
3. **数据集 diff 器** —— 把版本差异精确分成新增/删除/内容修改/元数据修改四类
4. **坏题的三条处理路径** —— 各自对历史可比性的影响，以及「交集重算」
5. **确定性抽样的稳定性** —— 为什么「打乱取前 N」在扩样本时会全变
6. **分层抽样的 smoke 子集** —— 偏向覆盖度而不是代表性

> 心智模型：**评测结果是一个定义在特定任务集上的量。
> 任务集变了，你估计的就是另一个总体参数——直接比较是无意义的。**

## 0 · 环境与示例数据集

In [ ]:
import os, json, math, hashlib, shutil, random, itertools
from collections import Counter, defaultdict

import numpy as np

TMP = os.path.abspath('./_eval_tmp')
if os.path.exists(TMP):
    shutil.rmtree(TMP)
os.makedirs(TMP, exist_ok=True)

def stable_hash(obj, n=8):
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()[:n]

def content_sha(task):
    # 只对**内容字段**取哈希——tags/meta 变化不算内容变化
    return stable_hash({k: task[k] for k in ('input', 'target')}, 12)

def make_task(task_id, inp, target, tags):
    t = {'task_id': task_id, 'input': inp, 'target': target, 'tags': tags}
    t['content_sha'] = content_sha(t)
    return t

V1 = [
    make_task('billing-001', '订单 A 的退款金额是多少？', '120', ['core', 'easy', 'billing']),
    make_task('billing-002', '用户能否取消已发货订单？', 'no',  ['core', 'medium', 'billing']),
    make_task('billing-003', '优惠券叠加规则是什么？',   'stack_max_1', ['hard', 'billing']),
    make_task('flight-001',  '改签手续费怎么算？',       'fee_10pct', ['core', 'medium', 'flight']),
    make_task('flight-002',  '超售时的补偿标准？',       'comp_400', ['hard', 'flight']),
    make_task('flight-003',  '婴儿票是否占座？',         'no', ['core', 'easy', 'flight']),
]
print(f'V1 共 {len(V1)} 条')
for t in V1[:2]:
    print(' ', {k: t[k] for k in ('task_id', 'content_sha', 'tags')})
assert len({t['task_id'] for t in V1}) == len(V1), 'task_id 必须唯一'
print('\n✅ 每条样本有两个标识：**稳定的 task_id**（永不改变）与 **content_sha**（用来检测内容变化）。')

## 1 · spec 的规范化：填默认值 → 迁移 → 再算指纹

**关键点**：指纹是「规范化之后的 spec」的哈希。
这样「显式写了默认值」和「没写」会算出同一个指纹 —— 这正是你想要的语义。

In [ ]:
SCHEMA_VERSION = 3

DEFAULTS = {
    'model': {'temperature': 0.0, 'seed': 0, 'max_tokens': 1024},
    # 注意 retries 默认是 0 —— 保持「旧 spec 没写就是不重试」的语义（讲解第 5 节）
    'budget': {'max_attempts': 1, 'timeout_s': 30, 'retries': 0},
    'runner': {'concurrency': 4, 'cache': True},
    'scorer': {'kind': 'exact_match', 'version': 'v1'},
}

def deep_fill(d, defaults):
    out = json.loads(json.dumps(d))
    for k, v in defaults.items():
        if isinstance(v, dict):
            out[k] = deep_fill(out.get(k, {}), v)
        else:
            out.setdefault(k, v)
    return out

def migrate(spec):
    """把旧 schema 升到当前版本。必须是纯函数且幂等。"""
    s = json.loads(json.dumps(spec))
    v = s.get('schema_version', 1)
    if v < 2:
        # v1 → v2: timeout 从毫秒改成秒时，**加新字段**而不是改旧字段的语义
        if 'timeout_ms' in s.get('budget', {}):
            s['budget']['timeout_s'] = s['budget'].pop('timeout_ms') / 1000.0
        v = 2
    if v < 3:
        # v2 → v3: dataset 从裸字符串变成三元组
        if isinstance(s.get('dataset'), str):
            s['dataset'] = {'id': s['dataset'], 'version': 'v1', 'sha': None}
        v = 3
    s['schema_version'] = SCHEMA_VERSION
    return s

def normalize(spec):
    return deep_fill(migrate(spec), DEFAULTS)

def fingerprint(spec):
    return stable_hash(normalize(spec), 8)

# 三份"看起来不同"但语义相同的 spec
spec_minimal = {'name': 'demo', 'dataset': {'id': 'qa', 'version': 'v1', 'sha': 'abc'},
                'model': {'id': 'm1'}}
spec_explicit = {'name': 'demo', 'dataset': {'id': 'qa', 'version': 'v1', 'sha': 'abc'},
                 'model': {'id': 'm1', 'temperature': 0.0, 'seed': 0, 'max_tokens': 1024},
                 'budget': {'max_attempts': 1, 'timeout_s': 30, 'retries': 0},
                 'runner': {'concurrency': 4, 'cache': True},
                 'scorer': {'kind': 'exact_match', 'version': 'v1'},
                 'schema_version': 3}
spec_old = {'name': 'demo', 'dataset': 'qa', 'model': {'id': 'm1'},
            'budget': {'timeout_ms': 30000}, 'schema_version': 1}

print('最简写法   指纹:', fingerprint(spec_minimal))
print('全写默认值 指纹:', fingerprint(spec_explicit))
assert fingerprint(spec_minimal) == fingerprint(spec_explicit)
print('→ 一致 ✓（写不写默认值不影响语义，因此不该影响指纹）')

# 迁移必须幂等
assert migrate(migrate(spec_old)) == migrate(spec_old)
print('\n旧 spec 迁移后:', json.dumps(migrate(spec_old)['budget'], ensure_ascii=False))
print('迁移幂等性 ✓')
print('\n✅ 指纹在规范化之后算——否则同一语义的新旧格式会算出不同指纹，历史结果又对不上了。')

In [ ]:
# 反例：如果新字段的默认值改变了旧 spec 的行为
BAD_DEFAULTS = json.loads(json.dumps(DEFAULTS)); BAD_DEFAULTS['budget']['retries'] = 2

def normalize_bad(spec):
    return deep_fill(migrate(spec), BAD_DEFAULTS)

old_semantics = normalize(spec_old)['budget']['retries']
new_semantics = normalize_bad(spec_old)['budget']['retries']
print(f'旧 spec 当时的行为: retries=0')
print(f'  正确的默认值 → retries={old_semantics}   （语义保持）')
print(f'  错误的默认值 → retries={new_semantics}   ← **所有历史 spec 的错误率都被悄悄改了**')
assert old_semantics == 0 and new_semantics == 2
print('\n✅ 新字段的默认值必须让旧 spec 的行为**保持不变**。')
print('   这条错了不会报错，只会让历史与现在的数字失去可比性。')

## 2 · 内容哈希 vs 稳定 ID：改一个错别字会怎样

In [ ]:
# 场景：修正 billing-003 的一个错别字（内容变了，但它还是同一道题）
V1_FIXED = json.loads(json.dumps(V1))
for t in V1_FIXED:
    if t['task_id'] == 'billing-003':
        t['input'] = '优惠券的叠加规则是什么？'      # 加了一个「的」
        t['content_sha'] = content_sha(t)

# 方案 A（❌）：用内容哈希当 ID
def id_by_content(task):
    return content_sha(task)

# 方案 B（✓）：稳定 ID
def id_stable(task):
    return task['task_id']

old_ids_A = {id_by_content(t) for t in V1}
new_ids_A = {id_by_content(t) for t in V1_FIXED}
old_ids_B = {id_stable(t) for t in V1}
new_ids_B = {id_stable(t) for t in V1_FIXED}

print(f'内容哈希当 ID: 修正前后能对上的样本 {len(old_ids_A & new_ids_A)} / {len(V1)}')
print(f'稳定 ID:       修正前后能对上的样本 {len(old_ids_B & new_ids_B)} / {len(V1)}')
assert len(old_ids_A & new_ids_A) == len(V1) - 1
assert len(old_ids_B & new_ids_B) == len(V1)
print('\n✅ 用内容哈希当 ID：改一个字，这条样本的历史结果就永远对不上了。')
print('   用稳定 ID：ID 不变，历史结果照样对得上；而 content_sha 单独告诉你「内容变过」。')
print('   → **content_sha 用来检测变化，不用来当 ID。**')

## 3 · 数据集 diff 器：四类变化，四种影响

In [ ]:
def dataset_diff(old, new):
    """把两个版本的差异精确分成四类。这四类对「历史结果能不能用」的影响完全不同。"""
    o = {t['task_id']: t for t in old}
    n = {t['task_id']: t for t in new}
    added = sorted(set(n) - set(o))
    removed = sorted(set(o) - set(n))
    common = set(o) & set(n)
    content_changed = sorted(i for i in common if o[i]['content_sha'] != n[i]['content_sha'])
    meta_changed = sorted(i for i in common
                          if o[i]['content_sha'] == n[i]['content_sha']
                          and o[i].get('tags') != n[i].get('tags'))
    return {'added': added, 'removed': removed,
            'content_changed': content_changed, 'meta_changed': meta_changed}

# 构造 v2：新增 2 条、剔除 1 条坏题、修正 1 条内容、改 1 条标签
V2 = [t for t in json.loads(json.dumps(V1)) if t['task_id'] != 'flight-002']   # 剔除坏题
for t in V2:
    if t['task_id'] == 'billing-003':
        t['input'] = '优惠券的叠加规则是什么？'; t['content_sha'] = content_sha(t)
    if t['task_id'] == 'flight-001':
        t['tags'] = ['core', 'hard', 'flight']            # 难度重标：medium → hard
V2 += [make_task('billing-004', '部分退款如何处理？', 'partial_ok', ['core', 'medium', 'billing']),
       make_task('flight-004',  '宠物托运费用？',     'pet_200',   ['hard', 'flight'])]

d = dataset_diff(V1, V2)
print(f"{'变化类型':<18}{'样本':<40}{'历史结果还能用吗'}")
IMPACT = {'added': '不适用（本来就没有）', 'removed': '保留但不再进入聚合',
          'content_changed': '**该样本的历史结果作废**', 'meta_changed': '能用（切片口径变了）'}
for k in ['added', 'removed', 'content_changed', 'meta_changed']:
    print(f'{k:<18}{str(d[k]):<40}{IMPACT[k]}')

assert d['added'] == ['billing-004', 'flight-004']
assert d['removed'] == ['flight-002']
assert d['content_changed'] == ['billing-003']
assert d['meta_changed'] == ['flight-001']
print('\n✅ 四类变化被精确分开——而这四类对历史可比性的影响完全不同。')
print('   注意 meta_changed：内容没变但难度标签变了，**切片口径变了但总分仍可比**。')

In [ ]:
# 版本号该怎么走：由 diff 自动推导
def suggest_version(old_ver, diff):
    major, minor = (int(x) for x in old_ver.lstrip('v').split('.')) if '.' in old_ver \
        else (int(old_ver.lstrip('v')), 0)
    breaking = bool(diff['removed'] or diff['content_changed'])
    if breaking:
        return f'v{major + 1}'
    if diff['added'] or diff['meta_changed']:
        return f'v{major}.{minor + 1}'
    return old_ver

print('本次 diff 建议的版本号:', suggest_version('v1', d))
assert suggest_version('v1', d) == 'v2'
assert suggest_version('v1', {'added': ['x'], 'removed': [], 'content_changed': [],
                              'meta_changed': []}) == 'v1.1'
assert suggest_version('v1', {'added': [], 'removed': [], 'content_changed': [],
                              'meta_changed': []}) == 'v1'
print('  只新增        → v1.1（次版本，历史全部有效）')
print('  剔除或改内容  → v2  （主版本，分母变了）')
print('\n✅ 版本号不是拍脑袋写的，是由 diff 推导出来的——这让「分母变了」这件事在文件名里就能看见。')

## 4 · 坏题的三条处理路径，以及「交集重算」

In [ ]:
rng = np.random.default_rng(0)
# 模拟：模型在 V1 上的逐题得分（flight-002 是坏题，谁做都错）
SCORES_V1 = {'billing-001': 1.0, 'billing-002': 1.0, 'billing-003': 0.0,
             'flight-001': 1.0, 'flight-002': 0.0, 'flight-003': 1.0}

def aggregate(scores, tasks, exclude_tag='excluded_from_main'):
    ids = [t['task_id'] for t in tasks if exclude_tag not in t.get('tags', [])]
    vals = [scores[i] for i in ids if i in scores]
    return (sum(vals) / len(vals)) if vals else float('nan'), len(vals)

# 路径 1：剔除并发 v2（分母变了）
V2_removed = [t for t in json.loads(json.dumps(V1)) if t['task_id'] != 'flight-002']
# 路径 2：保留但标记 excluded_from_main（分母也变了，但样本还在）
V2_marked = json.loads(json.dumps(V1))
for t in V2_marked:
    if t['task_id'] == 'flight-002':
        t['tags'] = t['tags'] + ['excluded_from_main']

base, n_base = aggregate(SCORES_V1, V1)
rem, n_rem = aggregate(SCORES_V1, V2_removed)
mark, n_mark = aggregate(SCORES_V1, V2_marked)
print(f"{'方案':<26}{'分数':>8}{'分母':>6}")
print(f'{"v1 原样":<26}{base:>8.1%}{n_base:>6}')
print(f'{"路径1 剔除并发 v2":<26}{rem:>8.1%}{n_rem:>6}')
print(f'{"路径2 保留但标记排除":<26}{mark:>8.1%}{n_mark:>6}')
assert rem == mark, '两条路径的主指标相同'
assert rem > base, '剔除一道谁都做错的坏题，分数必然上升'
print(f'\n⚠️ 分数从 {base:.1%} 涨到 {rem:.1%}——**这 16.7 个点完全来自分母变化，模型一点没变**。')
print('   这就是为什么剔除必须走主版本号：让「分母变了」在文件名里就能看见。')

In [ ]:
# 唯一正确的跨版本比较方式：在交集上重算两边
def intersect_compare(scores_a, tasks_a, scores_b, tasks_b):
    ids_a = {t['task_id'] for t in tasks_a}
    ids_b = {t['task_id'] for t in tasks_b}
    common = sorted(ids_a & ids_b)
    va = [scores_a[i] for i in common if i in scores_a]
    vb = [scores_b[i] for i in common if i in scores_b]
    return {'n_common': len(common),
            'a': sum(va) / len(va) if va else float('nan'),
            'b': sum(vb) / len(vb) if vb else float('nan')}

# 新模型在 v2 上的得分（把 billing-003 做对了，其余不变）
SCORES_V2 = dict(SCORES_V1); SCORES_V2['billing-003'] = 1.0
SCORES_V2.update({'billing-004': 1.0, 'flight-004': 0.0})

naive_a, _ = aggregate(SCORES_V1, V1)
naive_b, _ = aggregate(SCORES_V2, V2)
print(f'朴素比较: v1 上 {naive_a:.1%} → v2 上 {naive_b:.1%}，看起来涨了 {naive_b-naive_a:+.1%}')

ic = intersect_compare(SCORES_V1, V1, SCORES_V2, V2)
print(f'\n交集重算（{ic["n_common"]} 条共同样本）: {ic["a"]:.1%} → {ic["b"]:.1%}，'
      f'真实提升 {ic["b"]-ic["a"]:+.1%}')
assert ic['n_common'] < len(V1) and ic['n_common'] < len(V2)
assert abs((ic['b'] - ic['a']) - (naive_b - naive_a)) > 1e-6
print('\n✅ 朴素比较把「分母变化」和「模型提升」混在了一起；')
print('   交集重算只在**两个版本都有的样本**上比较，这才是唯一可解释的跨版本对比。')
print('   注意：billing-003 内容变过，严格来说也该排除——练习 2 会处理这个细节。')

## 5 · 确定性抽样：为什么「打乱取前 N」在扩样本时会全变

In [ ]:
def sample_shuffle(tasks, n, seed):
    """打乱后取前 N。确定性有（同 seed 同结果），但对任务集增长完全不稳定。"""
    ids = sorted(t['task_id'] for t in tasks)
    rnd = random.Random(seed)
    rnd.shuffle(ids)
    return sorted(ids[:n])

def sample_hash_topn(tasks, n, seed):
    """按 hash(task_id + seed) 排序取前 N。看起来更稳，但**只要 N 固定就仍然不稳**。"""
    scored = sorted(tasks, key=lambda t: stable_hash([t['task_id'], seed], 16))
    return sorted(t['task_id'] for t in scored[:n])

def sample_rate(tasks, rate, seed):
    """✓ 按**比例**选：每条样本独立判定，与任务集里有哪些别的样本无关。
    这是唯一能让「已有样本的入选与否永不改变」的做法。"""
    keep = []
    for t in tasks:
        h = int(stable_hash([t['task_id'], seed], 16), 16)
        if (h % 10000) < rate * 10000:
            keep.append(t['task_id'])
    return sorted(keep)

BIG = V1 + [make_task(f'gen-{i:03d}', f'q{i}', str(i), ['core']) for i in range(40)]
BIG_PLUS = BIG + [make_task(f'x{i:04d}', f'q{i}', str(i), ['core']) for i in range(200)]

a1, b1 = set(sample_shuffle(BIG, 15, 0)), set(sample_shuffle(BIG_PLUS, 15, 0))
a2, b2 = set(sample_hash_topn(BIG, 15, 0)), set(sample_hash_topn(BIG_PLUS, 15, 0))
a3, b3 = set(sample_rate(BIG, 0.33, 0)), set(sample_rate(BIG_PLUS, 0.33, 0))

print(f"{'做法':<24}{'原始规模':>10}{'扩后规模':>10}{'原有样本的入选是否改变':>26}")
print(f'{"打乱取前 N":<24}{len(a1):>10}{len(b1):>10}'
      f'{f"变了 {len(a1 - b1)} 条":>26}')
print(f'{"hash 排序取前 N":<24}{len(a2):>10}{len(b2):>10}'
      f'{f"变了 {len(a2 - b2)} 条":>26}')
print(f'{"按比例选（rate=0.33）":<24}{len(a3):>10}{len(b3):>10}'
      f'{f"变了 {len(a3 - b3)} 条":>26}')

assert len(a1 - b1) > 0 and len(a2 - b2) > 0, '固定 N 的两种做法都会掉样本'
assert a3 <= b3, '按比例选：原有的入选样本必须全部保留'
assert len(a3 - b3) == 0
print('\n✅ 关键结论：**只要 N 固定，任务集增长时抽中的那批就必然会变**——')
print('   这不是实现问题，「从 46 条抽 15」和「从 246 条抽 15」本来就是两个不同的均匀样本。')
print('   按比例选则完全稳定：每条样本独立判定，与任务集里有哪些别的样本无关。')
print(f'   代价是规模会随任务集增长（{len(a3)} → {len(b3)} 条），需要配合分层保底来控制。')

In [ ]:
# 分层抽样：smoke 子集要偏向**覆盖度**而不是代表性
def stratified_smoke(tasks, per_stratum=2, key=lambda t: (t['tags'][1] if len(t['tags']) > 1
                                                          else 'na'), seed=0):
    """按难度分层，每层保底 per_stratum 条。目标是覆盖，不是代表性。"""
    by = defaultdict(list)
    for t in tasks:
        by[key(t)].append(t)
    out = []
    for k in sorted(by):
        ranked = sorted(by[k], key=lambda t: stable_hash([t['task_id'], seed], 16))
        out += [t['task_id'] for t in ranked[:per_stratum]]
    return sorted(out)

MIXED = V1 + [make_task(f'g{i:02d}', f'q{i}', str(i),
                        ['core', ['easy', 'medium', 'hard'][i % 3]]) for i in range(30)]
smoke = stratified_smoke(MIXED, per_stratum=3)
cover = Counter(next((tt for tt in t['tags'] if tt in ('easy', 'medium', 'hard')), 'na')
                for t in MIXED if t['task_id'] in smoke)
print('smoke 子集:', smoke)
print('各难度覆盖:', dict(cover))
assert all(v >= 3 for k, v in cover.items() if k != 'na'), '每个难度层都要有保底名额'
assert len(smoke) < len(MIXED) / 3
print(f'\n✅ {len(smoke)} 条 smoke 覆盖了全部难度层，规模只有全集的 {len(smoke)/len(MIXED):.0%}。')
print('   smoke 的目标不是准确估计分数，是**抓住明显的退化**——所以覆盖度优先于代表性。')

## ✏️ 练习 1：数据集健康检查

实现 `dataset_health(tasks)`：返回一个字典，包含
`n`（样本数）、`dup_ids`（重复的 task_id 列表）、`dup_content`（内容完全相同的 content_sha 列表）、
`stale_sha`（`content_sha` 与重算结果不一致的 task_id 列表）、`ok`（三者都为空则 True）。

In [ ]:
def dataset_health(tasks):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
h = dataset_health(V1)
assert h['ok'] is True and h['n'] == len(V1)

bad = json.loads(json.dumps(V1))
bad.append(dict(bad[0]))                                    # 重复 ID
bad[1]['input'] = '改了内容但没更新 content_sha'             # stale sha
h2 = dataset_health(bad)
print('健康的数据集:', {k: v for k, v in h.items() if k != 'n'})
print('有问题的数据集:', {k: v for k, v in h2.items() if k != 'n'})
assert h2['ok'] is False
assert h2['dup_ids'] == ['billing-001']
assert h2['stale_sha'] == ['billing-002']
print('✅ 练习 1 通过：这三项检查应当在 CI 里对每个数据集版本跑一遍——')
print('   重复 ID 会让聚合分母出错，stale sha 会让 diff 器漏报内容变化。')

## ✏️ 练习 2：严格的交集比较

实现 `strict_intersect(scores_a, tasks_a, scores_b, tasks_b)`：
在 `intersect_compare` 的基础上，**额外排除 `content_sha` 变过的样本**
（内容变了就不是同一道题）。返回
`{'n_common', 'n_excluded_content_change', 'a', 'b', 'delta'}`。

In [ ]:
def strict_intersect(scores_a, tasks_a, scores_b, tasks_b):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
si = strict_intersect(SCORES_V1, V1, SCORES_V2, V2)
loose = intersect_compare(SCORES_V1, V1, SCORES_V2, V2)
print('宽松交集:', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in loose.items()})
print('严格交集:', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in si.items()})
assert si['n_excluded_content_change'] == 1, 'billing-003 内容变过，必须被排除'
assert si['n_common'] == loose['n_common'] - 1
assert abs(si['delta'] - (si['b'] - si['a'])) < 1e-12
print('✅ 练习 2 通过：billing-003 内容变过，虽然 ID 相同，但它已经不是同一道题了。')
print('   把它算进交集，就把「题变简单了」误记成了「模型变强了」。')

## ✏️ 练习 3：抽样稳定性的量化

实现 `retention_curve(select_fn, tasks, growth_list, seed=0)`：
`select_fn(tasks, seed)` 返回被选中的 ID 集合。对每个「新增样本数」，
返回 `(新增数, 原始入选样本的保留比例)`——即
`|原始入选 ∩ 扩后入选| / |原始入选|`。

**这个量才是 smoke 子集真正需要的稳定性**：不是「样本数不变」，
而是「原来入选的那些还在不在」。

In [ ]:
def retention_curve(select_fn, tasks, growth_list, seed=0):
    # TODO：新增的样本用 make_task(f'y{i:04d}', ...) 生成
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
GROWTH = [10, 50, 200]
r_shuffle = retention_curve(lambda ts, sd: set(sample_shuffle(ts, 15, sd)), BIG, GROWTH)
r_topn = retention_curve(lambda ts, sd: set(sample_hash_topn(ts, 15, sd)), BIG, GROWTH)
r_rate = retention_curve(lambda ts, sd: set(sample_rate(ts, 0.33, sd)), BIG, GROWTH)
print(f"{'新增样本数':>12}{'打乱取前N':>13}{'hash取前N':>13}{'按比例选':>12}")
for (g, s1), (_, s2), (_, s3) in zip(r_shuffle, r_topn, r_rate):
    print(f'{g:>12}{s1:>13.0%}{s2:>13.0%}{s3:>12.0%}')
assert all(abs(s - 1.0) < 1e-12 for _, s in r_rate), '按比例选的保留率必须恒为 100%'
assert r_shuffle[-1][1] < 0.5 and r_topn[-1][1] < 0.5
print('✅ 练习 3 通过：两种「固定 N」的做法在任务集翻几倍后保留率都掉到一半以下，')
print('   而按比例选恒为 100%——因为每条样本的入选是独立判定的。')
print('   → CI 的 smoke 子集应当按比例选 + 分层保底，而不是固定 N。')

## ✏️ 练习 4：spec 迁移的往返一致性

实现 `migration_roundtrip_ok(old_specs)`：对每个旧 spec 检查三条性质，
全部满足返回 True：① 迁移幂等 `migrate(migrate(s)) == migrate(s)`；
② 迁移后 `schema_version == SCHEMA_VERSION`；
③ 迁移不改变指纹语义——即 `fingerprint(s) == fingerprint(migrate(s))`。

In [ ]:
def migration_roundtrip_ok(old_specs):
    # TODO：返回 (是否全部通过, 失败的 spec 下标列表)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
OLD_SPECS = [
    spec_old,
    {'name': 'x', 'dataset': 'qa', 'model': {'id': 'm2'}, 'schema_version': 2},
    {'name': 'y', 'dataset': {'id': 'qa', 'version': 'v2', 'sha': 'z'},
     'model': {'id': 'm3'}, 'schema_version': 3},
]
ok, failed = migration_roundtrip_ok(OLD_SPECS)
print('全部通过:', ok, '| 失败下标:', failed)
assert ok is True and failed == []

broken = [{'name': 'z', 'dataset': 'qa', 'model': {'id': 'm4'}}]     # 无 schema_version → v1
ok2, _ = migration_roundtrip_ok(broken)
assert ok2 is True, '缺 schema_version 应按 v1 处理并能正常迁移'
print('✅ 练习 4 通过：第 3 条（迁移不改变指纹）是最容易被忽略的——')
print('   它保证了「历史 spec 迁移到新格式后，仍然指向同一次运行」。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def dataset_health(tasks):
    ids = [t['task_id'] for t in tasks]
    dup_ids = sorted({i for i, c in Counter(ids).items() if c > 1})
    shas = [t['content_sha'] for t in tasks]
    dup_content = sorted({s for s, c in Counter(shas).items() if c > 1})
    stale = sorted({t['task_id'] for t in tasks if t['content_sha'] != content_sha(t)})
    return {'n': len(tasks), 'dup_ids': dup_ids, 'dup_content': dup_content,
            'stale_sha': stale,
            'ok': not (dup_ids or dup_content or stale)}

In [ ]:
# 练习 2 参考答案
def strict_intersect(scores_a, tasks_a, scores_b, tasks_b):
    a = {t['task_id']: t for t in tasks_a}
    b = {t['task_id']: t for t in tasks_b}
    common = set(a) & set(b)
    changed = {i for i in common if a[i]['content_sha'] != b[i]['content_sha']}
    usable = sorted(common - changed)
    va = [scores_a[i] for i in usable if i in scores_a]
    vb = [scores_b[i] for i in usable if i in scores_b]
    ma = sum(va) / len(va) if va else float('nan')
    mb = sum(vb) / len(vb) if vb else float('nan')
    return {'n_common': len(usable), 'n_excluded_content_change': len(changed),
            'a': ma, 'b': mb, 'delta': mb - ma}

In [ ]:
# 练习 3 参考答案
def retention_curve(select_fn, tasks, growth_list, seed=0):
    base = set(select_fn(tasks, seed))
    out = []
    for g in growth_list:
        grown = tasks + [make_task(f'y{i:04d}', f'q{i}', str(i), ['core']) for i in range(g)]
        cur = set(select_fn(grown, seed))
        out.append((g, len(base & cur) / len(base) if base else float('nan')))
    return out

In [ ]:
# 练习 4 参考答案
def migration_roundtrip_ok(old_specs):
    failed = []
    for i, s in enumerate(old_specs):
        m = migrate(s)
        if migrate(m) != m:
            failed.append(i); continue
        if m.get('schema_version') != SCHEMA_VERSION:
            failed.append(i); continue
        if fingerprint(s) != fingerprint(m):
            failed.append(i); continue
    return (len(failed) == 0, failed)

---
## 🧪 真实工程胶囊：数据集目录与 CI 检查

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 目录结构（讲解第 7 节）
# ══════════════════════════════════════════════════════════════════
# dataset/qa-tasks/v1/{tasks.jsonl, datasheet.md, MANIFEST.json}
# dataset/qa-tasks/v2/{tasks.jsonl, datasheet.md, MANIFEST.json, CHANGELOG.md}
# dataset/subsets/smoke.json
# spec/demo-qa.json      ← 只引用 (id, version, sha)，不内联任务内容

# MANIFEST.json:
{ "id": "qa-tasks", "version": "v2", "sha": "7d2e…", "n": 512,
  "created": "2026-03-01", "frozen": true,
  "expected_score_range": [0.30, 0.55] }     # ← 讲解第 6 节：极便宜的健康检查

# ══════════════════════════════════════════════════════════════════
# B. CI 里必跑的四条数据集检查
# ══════════════════════════════════════════════════════════════════
def ci_dataset_checks(version_dir):
    tasks = load_jsonl(f"{version_dir}/tasks.jsonl")
    man = json.load(open(f"{version_dir}/MANIFEST.json"))
    # 1. 冻结版本不许被改
    if man.get("frozen"):
        assert stable_hash(tasks, 12) == man["sha"], "已冻结版本被修改了"
    # 2. 结构健康（练习 1）
    h = dataset_health(tasks)
    assert h["ok"], f"数据集结构有问题: {h}"
    # 3. 有 datasheet，且写了预期分数区间
    assert os.path.exists(f"{version_dir}/datasheet.md")
    assert "expected_score_range" in man
    # 4. 版本号与上一版的 diff 一致（讲解第 3 节）
    prev = load_prev_version(version_dir)
    if prev:
        assert man["version"] == suggest_version(prev["version"], dataset_diff(prev["tasks"], tasks))

# ══════════════════════════════════════════════════════════════════
# C. 跨版本比较：唯一正确的做法
# ══════════════════════════════════════════════════════════════════
# ✗ 直接比 v1 的分数和 v2 的分数
# ✓ strict_intersect(scores_v1, tasks_v1, scores_v2, tasks_v2)
#   并在报告里写明: "在 N 条共同且内容未变的样本上比较"
# 报告模板:
#   v1 (n=512): 41.2%    v2 (n=498): 44.7%    ← 不可直接比较
#   交集 (n=487, 排除 11 条内容变更): 41.0% → 43.1%   ← 真实提升 +2.1pp

# ══════════════════════════════════════════════════════════════════
# D. datasheet.md 的七个必答问题（讲解第 6 节）
# ══════════════════════════════════════════════════════════════════
# 1. 样本从哪来（污染风险）      2. 创建/采集时间（时间截断检验）
# 3. 谁标注的、规范是什么        4. 已知坏题清单与剔除理由
# 5. 不可能任务的比例            6. 判分器与它的 alpha/beta
# 7. **预期的合理分数区间**      ← 最便宜、最实用的一条
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| spec 是纯数据 | 换来可哈希、可 diff、可生成；prompt 的**哈希**必须进 spec | 全课 |
| 稳定 ID | ID 分配一次永不改；content_sha 用来检测变化，不当 ID | 数据集设计 |
| 版本语义 | 新增走次版本，修改/删除走主版本——因为分母变了 | 发布新版 |
| 交集重算 | 跨版本比较的唯一正确做法，且要排除内容变过的样本 | 报告 |
| 确定性抽样 | 固定 N 必然不稳；smoke 子集要**按比例选 + 分层保底** | smoke 子集 |
| schema 迁移 | 只增字段、默认值保持旧行为、**指纹在规范化之后算** | 长期演化 |
| datasheet | 七个必答问题，其中「预期分数区间」最便宜最实用 | 每个数据集版本 |

下一模块：**02 · Runner 工程**——并发、限流、重试、超时、缓存、断点续跑、预算熔断，
以及那个「改了 prompt 却读到旧结果」的隐蔽 bug 到底怎么防。